In [45]:
from pathlib import Path

BASE_DIR = Path('../dataset/test-dataset')

In [46]:
from dataclasses import dataclass, field
import torch

@dataclass
class DetectedResult:
    model_id: str
    x1: float
    y1: float
    x2: float
    y2: float
    class_id: int
    conf: float

@dataclass
class Image:
    filename: str

    path: Path = field(init=False, repr=False)
    tensors: torch.Tensor | None = field(init=False, default=None)
    
    detected: list[DetectedResult] = field(init=False, default_factory=list)

    def __post_init__(self):
        self.path = BASE_DIR / self.filename


In [47]:
import os

def get_images():
    images: list[Image] = []

    for file in os.listdir(BASE_DIR):
        if file == '.DS_Store':
            continue
        images.append(Image(filename=file))
    
    return images

In [48]:
from torchvision import transforms
from PIL import Image as PILImage

TARGET_SIZE = 224

transform_pipeline = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def transform_image(image: Image):
    img = PILImage.open(image.path)
    image.tensor = transform_pipeline(img)

In [49]:
from torch.utils.data import Dataset

class ImageDataset(Dataset):
    def __init__(self, images: list[Image]) -> None:
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        transform_image(image)

        return str(image.path), image.tensors

In [50]:
from torch.utils.data import DataLoader

dataset = ImageDataset(images=get_images())

dataloader = DataLoader(
    dataset=dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [51]:
from transformers import AutoModel

local_model_dir = "../weights/dinov2_small"
embedding_model_id = "facebook/dinov2-small"

def load_embedding_model():
    model = AutoModel.from_pretrained(embedding_model_id)
    model.save_pretrained(local_model_dir)
    return model

def get_embedding_model_eval():
    if not os.path.exists(local_model_dir):
        return load_embedding_model()
    model = AutoModel.from_pretrained(local_model_dir, local_files_only=True)
    return model

## Извлечение признаков

In [52]:
from tqdm import tqdm
import numpy as np

embeddings_path = Path('../dataset/embeddings')

def extract_embeddings():
    if os.path.exists(embeddings_path / "image_embeddings_dinov2_small.npy"):
        return np.load(embeddings_path / "image_embeddings_dinov2_small.npy")

    device = torch.device('mps')

    all_embeddings = []

    model = get_embedding_model_eval()
    model.to(device)
    model.eval()

    for _, tensors in tqdm(dataloader, desc="Извлечение признаков"):
        tensors = tensors.to(device)
        with torch.no_grad():
            outputs = model(pixel_values=tensors)

        tokens = outputs.last_hidden_state[:, 0, :]
        embeddings = tokens.cpu().numpy()

        all_embeddings.append(embeddings)

    final_embeddings = np.vstack(all_embeddings)
    np.save(embeddings_path / "image_embeddings_dinov2_small.npy", final_embeddings)
    return final_embeddings

In [53]:
embeddings = extract_embeddings()
print(f"Форма {embeddings.shape}")

Форма (72, 384)


## Понижение размерности

In [54]:
from sklearn.decomposition import PCA

def dimensionality_reduction(n_components: int = 8):
    pca = PCA(n_components=n_components, random_state=42)
    reduced_embeddings = pca.fit_transform(embeddings)

    print(f"Форма {reduced_embeddings.shape}")
    print(f"Сохранение информации: {pca.explained_variance_ratio_.sum() * 100}")

    np.save(embeddings_path / f"image_embeddings_dinov2_small_{n_components}.npy", reduced_embeddings)

    return reduced_embeddings

In [55]:
reduced_embeddings = dimensionality_reduction(32)

Форма (72, 32)
Сохранение информации: 97.8763198852539


## Клатеризация по зонам

In [56]:
from sklearn.cluster import AgglomerativeClustering
import hdbscan

def agglomerative_clustering(n_clusters: int = 2):
    cluster_model = AgglomerativeClustering(linkage='ward', n_clusters=n_clusters)
    labels = cluster_model.fit_predict(reduced_embeddings)
    return labels

def hdbscan_clustering():
    cluster_model = hdbscan.HDBSCAN(
        min_cluster_size=4,
        min_samples=10,
        metric='euclidean',
        cluster_selection_epsilon=0.5
    )
    labels = cluster_model.fit_predict(reduced_embeddings)
    return labels

In [57]:
hdbscan_labels = hdbscan_clustering()
agglomerative_labels = agglomerative_clustering(n_clusters=len(set(hdbscan_labels)))

In [58]:
@dataclass
class CameraFrameBatch:
    images: tuple[Image] = field(default_factory=tuple)
    cameras: tuple[int] = field(default_factory=tuple)

    def __post_init__(self):
        if len(self.images) != len(self.cameras):
            raise ValueError("Невозможно установить происхождение кадров")

    def __getitem__(self, idx):
        return self.images[idx], self.cameras[idx]


In [59]:
camera_frame_batch = CameraFrameBatch(
    images=tuple(dataset.images),
    cameras=tuple(agglomerative_labels)
)

## Детектинг

In [60]:
img_paths: str = [img.path for img, _ in camera_frame_batch]

In [61]:
from ultralytics import YOLO

def detecting_hazard():
    model = YOLO("../weights/hazard/yolo26s.pt")
    results = model(img_paths)
    return results

In [62]:
hazard_result = detecting_hazard()


0: 640x640 3 machinerys, 294.1ms
1: 640x640 1 Person, 1 Safety Cone, 1 Safety Vest, 1 machinery, 294.1ms
2: 640x640 3 machinerys, 294.1ms
3: 640x640 1 Person, 2 machinerys, 294.1ms
4: 640x640 7 machinerys, 2 vehicles, 294.1ms
5: 640x640 1 machinery, 294.1ms
6: 640x640 1 machinery, 294.1ms
7: 640x640 2 machinerys, 294.1ms
8: 640x640 1 Hardhat, 1 Safety Cone, 1 machinery, 294.1ms
9: 640x640 2 machinerys, 294.1ms
10: 640x640 1 Person, 1 Safety Vest, 1 machinery, 294.1ms
11: 640x640 2 Hardhats, 2 Persons, 2 Safety Vests, 1 machinery, 294.1ms
12: 640x640 2 machinerys, 294.1ms
13: 640x640 1 Person, 2 Safety Vests, 2 machinerys, 294.1ms
14: 640x640 1 Hardhat, 1 NO-Safety Vest, 1 Person, 1 machinery, 294.1ms
15: 640x640 1 Safety Cone, 2 machinerys, 294.1ms
16: 640x640 3 machinerys, 294.1ms
17: 640x640 1 Person, 3 Safety Cones, 1 Safety Vest, 294.1ms
18: 640x640 (no detections), 294.1ms
19: 640x640 1 machinery, 294.1ms
20: 640x640 2 machinerys, 294.1ms
21: 640x640 1 Person, 1 machinery, 294.1m

In [63]:
@dataclass
class DetectedResult:
    model_id: str
    x1: float
    y2: float
    x2: float
    y2: float
    class_id: int
    conf: float

In [ ]:
for i, cls in enumerate(hazard_result):
    result = cls.boxes.data.cpu().numpy()

    img = camera_frame_batch.images[i]

    for x1, y1, x2, y2, conf, class_id in result:
        img.detected.append(
            DetectedResult(x1, y1, x2, y2, conf, class_id)
        )

        print(img)

Image(filename='1422_13_51_29_383011-2023-10-16.jpg', tensors=None, detected=[DetectedResult(model_id=np.float32(2436.0635), x1=np.float32(1053.9684), y2=np.float32(2688.0), x2=np.float32(1459.4989), class_id=np.float32(0.82786715), conf=np.float32(8.0))])
Image(filename='1422_13_51_29_383011-2023-10-16.jpg', tensors=None, detected=[DetectedResult(model_id=np.float32(2436.0635), x1=np.float32(1053.9684), y2=np.float32(2688.0), x2=np.float32(1459.4989), class_id=np.float32(0.82786715), conf=np.float32(8.0)), DetectedResult(model_id=np.float32(506.8184), x1=np.float32(475.29028), y2=np.float32(704.3498), x2=np.float32(1101.4749), class_id=np.float32(0.47367105), conf=np.float32(8.0))])
Image(filename='1422_13_51_29_383011-2023-10-16.jpg', tensors=None, detected=[DetectedResult(model_id=np.float32(2436.0635), x1=np.float32(1053.9684), y2=np.float32(2688.0), x2=np.float32(1459.4989), class_id=np.float32(0.82786715), conf=np.float32(8.0)), DetectedResult(model_id=np.float32(506.8184), x1=np